# 08c | Drills: Logistic regression and regularization

Goes with **08a and 08b**: converting between sigmoid and odds, log loss, sklearn Pipelines, AUC / PR-AUC, thresholds and precision / recall, L1 vs. L2, `smf.logit` and odds ratios.

**How to use this page:** There are no input boxes here. There are two kinds of problems:

- **Computation problems**: the code and its output are shown right below. First think about which function and formula to use, then read on for the answer;
- **Spoken problems** (marked 🗣 after the title): answer out loud first (30–60 seconds), then compare with the reference answer.

Problems on a page run in order, so variables and functions defined earlier can be reused later. Every problem assumes this has already been run:

```python
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, precision_score, recall_score, average_precision_score

X, y = make_classification(n_samples=3000, n_features=12, n_informative=6,
                           weights=[0.85, 0.15], random_state=8)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=8)
```

### Problem 1 · Sigmoid and odds ratio

What probability corresponds to log-odds of −1.2? What odds ratio corresponds to a coefficient of 0.7?

```python
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

print(round(sigmoid(-1.2), 4), round(np.exp(0.7), 4))
```

**Result:**

```text
0.2315 2.0138
```

> Probability = sigmoid(log-odds); odds ratio = $e^\beta$.

### Problem 2 · A coefficient's effect on probability depends on the baseline

Coefficient 0.7. With baseline log-odds of −2 and of 0, how much does the probability rise when x increases by 1 unit?

```python
for base in [-2.0, 0.0]:
    p0, p1 = sigmoid(base), sigmoid(base + 0.7)
    print(base, round(p0, 4), round(p1, 4), round(p1 - p0, 4))
```

**Result:**

```text
-2.0 0.1192 0.2142 0.095
0.0 0.5 0.6682 0.1682
```

> The odds are multiplied by about 2 in both cases, but the probability increments differ: the increase is largest when p is near 0.5 (close to β/4 = 0.175) and smaller when p is small.

### Problem 3 · Baseline log loss

If you predict the training set's share of positives for everyone, what is the log loss on the test set?

```python
def log_loss(y, p):
    return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))

base_rate = y_train.mean()
print(round(base_rate, 4), round(log_loss(y_test, base_rate), 4))
```

**Result:**

```text
0.1529 0.4284
```

> This is the best log loss achievable "with no features", and the model must at least beat it. With 15% positives it is about 0.42; with a 50/50 split it is ln 2 ≈ 0.693.

### Problem 4 · Pipeline: training and evaluation

Put standardization + logistic regression into a Pipeline, fit on the training set, and compute the test-set ROC-AUC, PR-AUC and log loss.

```python
pipe = Pipeline([
    ("scale", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000)),
])
pipe.fit(X_train, y_train)
proba = pipe.predict_proba(X_test)[:, 1]
print(round(roc_auc_score(y_test, proba), 4))
print(round(average_precision_score(y_test, proba), 4))
print(round(log_loss(y_test, proba), 4))
```

**Result:**

```text
0.9344
0.6793
0.2272
```

> Column 1 of `predict_proba` is the positive-class probability. Metric functions always take `(true values, predictions)`.

### Problem 5 · The accuracy trap

What is the accuracy of "predict everything negative"? And the model's accuracy at threshold 0.5?

```python
print(round((y_test == 0).mean(), 4))
print(round((pipe.predict(X_test) == y_test).mean(), 4))
```

**Result:**

```text
0.8467
0.9011
```

> Doing nothing already gives 85% accuracy, so under class imbalance accuracy carries almost no information.

### Problem 6 · Changing the threshold

Look at precision and recall at thresholds 0.5, 0.3 and 0.15.

```python
for th in [0.5, 0.3, 0.15]:
    pred = (proba >= th).astype(int)
    print(th, round(precision_score(y_test, pred), 3), round(recall_score(y_test, pred), 3))
```

**Result:**

```text
0.5 0.699 0.623
0.3 0.632 0.848
0.15 0.508 0.942
```

> The lower the threshold, the higher the recall and the lower the precision. The model is unchanged; only the decision rule changed.

### Problem 7 · Precision and recall by hand

At threshold 0.3, count TP, FP and FN yourself, compute precision and recall, and check against sklearn.

```python
pred = (proba >= 0.3).astype(int)
tp = ((pred == 1) & (y_test == 1)).sum()
fp = ((pred == 1) & (y_test == 0)).sum()
fn = ((pred == 0) & (y_test == 1)).sum()
print(tp, fp, fn)
print(round(tp / (tp + fp), 3), round(tp / (tp + fn), 3))
print(round(precision_score(y_test, pred), 3), round(recall_score(y_test, pred), 3))
```

**Result:**

```text
117 68 21
0.632 0.848
0.632 0.848
```

> precision = TP / (TP + FP) (of the flagged cases, how many are real); recall = TP / (TP + FN) (of the real cases, how many were found).

### Problem 8 · What AUC means

Pick a random positive and a random negative: the probability that the model scores the positive higher. Count it directly and compare with `roc_auc_score`.

```python
pos = proba[y_test == 1]
neg = proba[y_test == 0]
wins = [(p > neg).mean() + 0.5 * (p == neg).mean() for p in pos]
print(round(np.mean(wins), 4), round(roc_auc_score(y_test, proba), 4))
```

**Result:**

```text
0.9344 0.9344
```

> The two are identical. This is the definition of AUC: it only looks at the ranking and does not depend on the threshold or the class ratio. A tied pair counts as half.

### Problem 9 · L1 vs. L2: counting zero coefficients

Only 4 of 30 features are useful. With `C = 0.1`, how many coefficients do L1 and L2 each push to exactly 0?

```python
Xs, ys = make_classification(n_samples=1000, n_features=30, n_informative=4, n_redundant=0, random_state=5)

def fit_coef(penalty, C):
    p = Pipeline([
        ("scale", StandardScaler()),
        ("model", LogisticRegression(penalty=penalty, solver="liblinear", C=C)),
    ])
    p.fit(Xs, ys)
    return p.named_steps["model"].coef_[0]

print((fit_coef("l1", 0.1) == 0).sum(), (fit_coef("l2", 0.1) == 0).sum())
```

**Result:**

```text
19 0
```

> L1 produces sparse solutions; L2 only shrinks and never sets coefficients to zero.

### Problem 10 · Smaller C, stronger regularization

Under L2 regularization, how does the sum of absolute coefficients change as C goes through 0.001, 0.01, 0.1, 1, 10?

```python
for C in [0.001, 0.01, 0.1, 1, 10]:
    print(C, round(abs(fit_coef("l2", C)).sum(), 3))
```

**Result:**

```text
0.001 0.617
0.01 2.736
0.1 6.168
1 8.387
10 8.84
```

> C is the inverse of λ: the smaller C, the heavier the penalty and the closer the coefficients are to 0 overall. C should be chosen by cross-validation (lesson 10).

### Problem 11 · Class weights

`class_weight="balanced"` gives errors on the minority class more weight. Compare precision and recall at threshold 0.5 with and without it.

```python
pipe_bal = Pipeline([
    ("scale", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000, class_weight="balanced")),
])
pipe_bal.fit(X_train, y_train)
for name, m in [("default", pipe), ("balanced", pipe_bal)]:
    pred = m.predict(X_test)
    print(name, round(precision_score(y_test, pred), 3), round(recall_score(y_test, pred), 3))
```

**Result:**

```text
default 0.699 0.623
balanced 0.524 0.942
```

> With weighting, recall rises sharply and precision falls, much like lowering the threshold. The price is that the probabilities from `predict_proba` are no longer calibrated true probabilities.

### Problem 12 · smf.logit and odds ratios

Simulate data where returning users (`returning = 1`) have log-odds of purchase 0.9 higher, and each extra page viewed adds 0.15. Estimate with `smf.logit` and report the odds ratios.

```python
rng = np.random.default_rng(12)
m = 3000
d = pd.DataFrame({"pages": rng.integers(1, 15, m), "returning": rng.integers(0, 2, m)})
logit_p = -3 + 0.15 * d["pages"] + 0.9 * d["returning"]
d["buy"] = (rng.random(m) < sigmoid(logit_p)).astype(int)

fit = smf.logit("buy ~ pages + returning", data=d).fit(disp=0)
pd.DataFrame({"coef": fit.params, "odds ratio": np.exp(fit.params)}).round(3)
```

**Result:**

|   | coef | odds ratio |
|---|---|---|
| Intercept | -3.079 | 0.046 |
| pages | 0.155 | 1.167 |
| returning | 0.925 | 2.522 |

*3 rows × 2 columns*

> The odds ratio for `returning` is about $e^{0.9} \approx 2.5$: returning users have about 2.5 times the odds of purchasing as new users; each extra page multiplies the odds by about $e^{0.15} \approx 1.16$.

### Problem 13 · Odds vs. probability 🗣

What odds correspond to a probability of 0.2? Can an odds ratio of 2 be described as "the probability doubles"?

**Reference answer:**

odds = p/(1 − p) = 0.2/0.8 = 0.25.

An odds ratio of 2 is not a doubling of probability. At a baseline probability of 0.2 (odds 0.25), doubling the odds to 0.5 gives a probability of 0.5/1.5 ≈ 0.33, not 0.4. Only when the baseline probability is small (say 1%) are odds and probability nearly equal, so that the odds ratio approximates the ratio of probabilities (the relative risk). When reporting to stakeholders, it's best to convert to concrete probabilities.

*Short version:* Odds of 0.25. An odds ratio of 2 is not a doubling of probability unless the outcome is rare; I'd translate it into probabilities at a typical baseline.

### Problem 14 · Why not train logistic regression with squared error? 🗣

Why does logistic regression use log loss instead of squared error like linear regression?

**Reference answer:**

(1) **It comes from the likelihood:** log loss is the negative log-likelihood of the Bernoulli distribution, so minimizing it is MLE, with good statistical properties; (2) **convexity:** for the logistic model, log loss is convex in the parameters, with no local optima; squared error wrapped around a sigmoid is not convex; (3) **it punishes confident mistakes:** predicting 0.01 when the truth is 1 gives a log loss of −log 0.01 ≈ 4.6, while squared error is at most 1, so the model never learns "don't be overconfident".

### Problem 15 · What happens with perfect separation? 🗣

One feature in the training data perfectly separates positives from negatives. What goes wrong with logistic regression?

**Reference answer:**

The MLE does not exist: pushing that feature's coefficient toward infinity keeps increasing the likelihood, so the algorithm fails to converge or returns huge coefficients and standard errors (statsmodels warns "Perfect separation").

Fixes: add regularization (sklearn uses L2 by default, so it won't diverge); check whether the feature "leaks" the outcome (e.g. using "refund date is not null" to predict "was refunded"); or use a penalized method such as Firth regression.

### Problem 16 · How to choose C 🗣

What is `C` in sklearn's `LogisticRegression`? How do you choose it?

**Reference answer:**

`C` is the inverse of the regularization strength (C = 1/λ), default 1.0. The smaller C, the heavier the penalty and the simpler the model (high bias, low variance); the larger C, the closer to unregularized MLE.

How to choose: cross-validate over a grid of C values (usually spaced by factors of 10, e.g. 0.001 to 100) and pick the one with the best validation metric (log loss or AUC); put all preprocessing in a Pipeline so each fold standardizes using only its training part.

### Problem 17 · How to set the threshold 🗣

Once the model outputs probabilities, is 0.5 always the best threshold? How do you choose?

**Reference answer:**

Not necessarily. 0.5 only makes sense when the two kinds of error cost the same and the probabilities are well calibrated.

How to choose: (1) by business cost: e.g. if missing a fraud case costs $500 and wrongly blocking a legitimate transaction costs $5, the threshold should be set very low; compute the total cost at each threshold on the validation set and take the minimum; (2) by constraint: e.g. the threshold with the highest recall subject to "precision at least 90%"; (3) by capacity: if only 200 cases a day can be reviewed manually, take the 200 highest scores. Choose the threshold on the validation set, never on the test set.

### Problem 18 · AUC = 0.5 and AUC < 0.5 🗣

What does an AUC of 0.5 mean? What if you get an AUC of 0.3?

**Reference answer:**

AUC = 0.5: the model ranks no better than random guessing; it has no discriminating power.

AUC = 0.3: the ranking is **reversed**; flip the predictions (use 1 − score) and the AUC becomes 0.7. In practice the more common cause is a bug: the label direction is flipped, column 0 of `predict_proba` was used, or the feature order differs between training and test. When you see an AUC clearly below 0.5, look for a bug first.

### Problem 19 · Lasso with highly correlated features 🗣

Two features are highly correlated and both related to the outcome. How does Lasso handle them? What's the problem?

**Reference answer:**

Lasso tends to keep only one of them and push the other to 0, and which one it keeps can be fairly arbitrary: a small change in the data and the selected variable switches. That makes interpreting "the variables Lasso selected" unstable.

Ridge spreads the weight across both variables, which is more stable but not sparse. **Elastic Net** (L1 + L2) gets both: it can push irrelevant variables to 0 while letting correlated variables enter as a group.

### Problem 20 · Why standardization must be fit on the training set only 🗣

Why put `StandardScaler` in a Pipeline instead of standardizing all the data first and then splitting?

**Reference answer:**

If you compute the mean and standard deviation on all the data first, information from the test set "leaks" into training, and test performance may be optimistic rather than an honest estimate for new data. For standardization alone this leakage is usually small; but the same mistake in missing-value imputation, feature selection or target encoding causes much bigger bias, so make a habit of "all preprocessing goes in the Pipeline".

In a Pipeline, `fit` computes the mean and standard deviation from training data only, and `predict` transforms new data with those same parameters, which matches what happens in production. In cross-validation the Pipeline also ensures that each fold uses only its own training part.

## Recap

| Task | Code |
|---|---|
| Probability / odds ratio | `1 / (1 + np.exp(-z))` / `np.exp(beta)` |
| Log loss | `-np.mean(y*np.log(p) + (1-y)*np.log(1-p))`; baseline is a constant prediction of the positive share |
| Modeling | `Pipeline([("scale", StandardScaler()), ("model", LogisticRegression(max_iter=1000))])` |
| Probabilities | `pipe.predict_proba(X)[:, 1]` |
| Ranking metrics | `roc_auc_score`, `average_precision_score`, arguments `(true, score)` |
| Threshold | `(proba >= th).astype(int)`, then `precision_score` / `recall_score` |
| Regularization | `penalty="l1", solver="liblinear"`; smaller `C` is stronger |
| Imbalance | `class_weight="balanced"` or tune the threshold |
| Inference | `smf.logit("y ~ x", data=d).fit(disp=0)`, `np.exp(params)` |